# 3.2 — Абляции, метрики по группам и out-of-distribution тесты

**Папка 3, подноутбук 2.** Анализ устойчивости и вклада компонентов: метрики по типам
грунта и режимам нагружения, абляции DPI-Flow и EVT-NeuralSSM, OOD-тесты (экстраполяция
«короткий → длинный горизонт» и удержанный регион слабых грунтов). Все рисунки и таблицы
— на английском.

## Окружение, данные и модели

In [1]:
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Найти корень репозитория по наличию pyproject.toml вверх по дереву."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
from IPython.display import display

from liquefaction_ai.viz import register_theme

register_theme()

# Если True — все фигуры сохраняются в results/figs (.html и .png)
SAVE_FIGS = True
DATA_DIR = REPO_ROOT / "data" / "demo_run"
MODELS_DIR = REPO_ROOT / "models"

import torch

from liquefaction_ai import load_population_artifact, prepare_benchmark_dataset
from liquefaction_ai.training import load_model_metadata, load_weights_into
from liquefaction_ai.models import DPIFlow, EVTNeuralSSM, GRUBaseline, RiskMLP, TCNBaseline
from liquefaction_ai.evaluation import collect_outputs, compute_metrics, english_metric_table

CLASS_REGISTRY = {"RiskMLP": RiskMLP, "GRUBaseline": GRUBaseline, "TCNBaseline": TCNBaseline,
                  "DPIFlow": DPIFlow, "EVTNeuralSSM": EVTNeuralSSM}
MODEL_NAMES = ["mlp_risk", "gru", "tcn", "dpi_flow", "evt_ssm"]

population, config = load_population_artifact(DATA_DIR)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
benchmark = prepare_benchmark_dataset(population, config, device)
test = benchmark["test"]


def load_trained(name):
    """Восстановить модель по сохранённым гиперпараметрам и весам."""
    hp, hist = load_model_metadata(MODELS_DIR, name)
    model = CLASS_REGISTRY[hp["model_type"]](**hp["model_kwargs"])
    load_weights_into(model, MODELS_DIR, name, device)
    return model, hp, hist
from liquefaction_ai.evaluation import (filter_split, grouped_metrics, is_holdout_region,
                                        run_quick_experiment, subsample_split)
from liquefaction_ai.constants import LOAD_DISPLAY_NAMES_EN, SOIL_DISPLAY_NAMES_EN
from liquefaction_ai.viz import bar, grouped_bar

## Metrics by soil type and loading regime (DPI-Flow)

In [2]:
dpi, hp, _ = load_trained("dpi_flow")
out = collect_outputs(dpi, test, config, device)
_, sample_df = compute_metrics("DPI-Flow", out, test, config)
sample_df["soil_en"] = sample_df["soil_type"].map(SOIL_DISPLAY_NAMES_EN)
sample_df["load_en"] = sample_df["load_mode"].map(LOAD_DISPLAY_NAMES_EN)
soil_metrics = grouped_metrics(sample_df, "soil_en")
load_metrics = grouped_metrics(sample_df, "load_en")
display(english_metric_table(soil_metrics).round(4))
grouped_bar(soil_metrics["soil_en"].tolist(),
            {"AUROC": soil_metrics["AUROC"].tolist(), "Trajectory RMSE": soil_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by soil type", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_soil").show()
grouped_bar(load_metrics["load_en"].tolist(),
            {"AUROC": load_metrics["AUROC"].tolist(), "Trajectory RMSE": load_metrics["mean_traj_rmse"].tolist()},
            title="DPI-Flow quality by loading regime", ylabel="Value (–)",
            save=SAVE_FIGS, fig_id="3_2_metrics_by_load").show()

,soil_en,N samples,Liquefaction rate,Mean predicted risk,Mean |ΔN_liq| (cycles),Mean trajectory RMSE,Mean interval width,AUROC
8,Silty sand,297,0.5320,0.3639,91.495499,0.0178,0.0407,0.9633
5,Medium sand,206,0.5437,0.3602,66.992996,0.0118,0.0448,0.9942
7,Sandy loam,180,0.5833,0.3916,92.877296,0.0154,0.0514,0.9726
2,Fine sand,157,0.5287,0.3604,58.246700,0.0119,0.0438,0.9925
4,Loam,115,0.4609,0.3532,93.813499,0.0197,0.0524,0.9763
1,Coarse sand,100,0.4800,0.2894,45.429100,0.0108,0.0364,0.9984
3,Gravelly sand,70,0.4714,0.2894,54.630402,0.0090,0.0391,0.9951
0,Clay,59,0.3051,0.2082,27.443501,0.0138,0.0389,0.9946
6,Peat,17,0.7059,0.4906,83.293404,0.0187,0.1027,1.0000


## Component ablations

In [3]:
sub = min(config.ablation_subset, 2400)
atrain = subsample_split(benchmark["train"], int(sub * 0.7), config.seed)
aval = subsample_split(benchmark["val"], int(sub * 0.15), config.seed + 1)
atest = subsample_split(benchmark["test"], int(sub * 0.15), config.seed + 2)
sd, pdim, qd = static_dim, prefix_dim, seq_dim = (benchmark["train"]["static"].shape[1],
    benchmark["train"]["prefix_summary"].shape[1], benchmark["train"]["seq_in"].shape[-1])
mcr, sl, pl = config.max_cycle_reference, config.seq_len, config.prefix_len
ablation_specs = [
    ("DPI-Flow (full)", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2, use_analytical_layer=True)),
    ("DPI-Flow w/o calibration", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=True)),
    ("DPI-Flow w/o probabilistic head", DPIFlow(sd, pdim, sl, pl, mcr, probabilistic=False, calibration_steps=2)),
    ("DPI-Flow w/o ODE layer", DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=0, use_analytical_layer=False)),
    ("EVT (full)", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)),
    ("EVT w/o trigger", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_trigger_head=False)),
    ("EVT w/o post-event dynamics", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, structured_post_event=False)),
    ("EVT w/o CRR damage", EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr, use_crr_damage=False)),
]
ablation_rows = [run_quick_experiment(n, m, atrain, aval, atest, epochs=config.ablation_epochs,
                                      config=config, device=device) for n, m in ablation_specs]
ablation_df = pd.DataFrame(ablation_rows).sort_values("Traj_RMSE")
display(english_metric_table(ablation_df)[["Model", "Trajectory RMSE", "AUROC", "Brier"]].round(4))
bar(ablation_df["model"], ablation_df["Traj_RMSE"], title="Ablations: trajectory RMSE (lower is better)",
    ylabel="Trajectory RMSE (–)", color="#6610f2", horizontal=True,
    save=SAVE_FIGS, fig_id="3_2_ablations_rmse").show()

[DPI-Flow (full)] эпоха 01 | обучение=-1.8729 | валидация=-2.2604


[DPI-Flow (full)] эпоха 02 | обучение=-2.4374 | валидация=-2.5423


[DPI-Flow w/o calibration] эпоха 01 | обучение=-1.6081 | валидация=-2.4196


[DPI-Flow w/o calibration] эпоха 02 | обучение=-2.4589 | валидация=-2.4389


[DPI-Flow w/o probabilistic head] эпоха 01 | обучение=-1.9803 | валидация=-2.3506


...

[EVT w/o post-event dynamics] эпоха 01 | обучение=-0.2833 | валидация=-1.2378


[EVT w/o post-event dynamics] эпоха 02 | обучение=-1.1401 | валидация=-1.0905


[EVT w/o CRR damage] эпоха 01 | обучение=5.8764 | валидация=-0.3498


[EVT w/o CRR damage] эпоха 02 | обучение=-0.6578 | валидация=-0.5733


,Model,Trajectory RMSE,AUROC,Brier
2,DPI-Flow w/o probabilistic head,0.0260,0.9664,0.1205
0,DPI-Flow (full),0.0267,0.9653,0.1116
1,DPI-Flow w/o calibration,0.0336,0.9733,0.0980
3,DPI-Flow w/o ODE layer,0.0577,0.9706,0.0800
6,EVT w/o post-event dynamics,0.1104,0.6261,0.2610
5,EVT w/o trigger,0.1478,0.5955,0.4913
7,EVT w/o CRR damage,0.1478,0.8263,0.4073
4,EVT (full),0.1478,0.9303,0.3834


## Out-of-distribution tests

In [4]:
train_meta = benchmark["train"]["meta"]; val_meta = benchmark["val"]["meta"]; test_meta = benchmark["test"]["meta"]
short_thr = float(train_meta["N_max"].quantile(0.60)); long_thr = float(test_meta["N_max"].quantile(0.80))
short_train = filter_split(benchmark["train"], train_meta["N_max"].to_numpy() <= short_thr)
short_val = filter_split(benchmark["val"], val_meta["N_max"].to_numpy() <= short_thr)
long_test = filter_split(benchmark["test"], test_meta["N_max"].to_numpy() >= long_thr)
e_thr = float(benchmark["meta"]["e"].quantile(0.75)); vs_thr = float(benchmark["meta"]["V_s"].quantile(0.25))
region_test = filter_split(benchmark["test"], is_holdout_region(test_meta, e_thr, vs_thr))
hold_train = filter_split(benchmark["train"], ~is_holdout_region(train_meta, e_thr, vs_thr))
hold_val = filter_split(benchmark["val"], ~is_holdout_region(val_meta, e_thr, vs_thr))
short_train = subsample_split(short_train, 1100, config.seed); short_val = subsample_split(short_val, 300, config.seed + 1)
long_test = subsample_split(long_test, 700, config.seed + 2)
hold_train = subsample_split(hold_train, 1100, config.seed + 3); hold_val = subsample_split(hold_val, 300, config.seed + 4)
region_test = subsample_split(region_test, 700, config.seed + 5)

def fresh(kind):
    if kind == "tcn": return TCNBaseline(sd, qd, 96)
    if kind == "dpi": return DPIFlow(sd, pdim, sl, pl, mcr, calibration_steps=2)
    return EVTNeuralSSM(sd, pdim, qd, sl, pl, mcr)

ood_rows = []
for disp, kind in [("TCN", "tcn"), ("DPI-Flow", "dpi"), ("EVT-NeuralSSM", "evt")]:
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), short_train, short_val, long_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "short→long"})
    ood_rows.append({**run_quick_experiment(disp, fresh(kind), hold_train, hold_val, region_test,
                     epochs=config.ablation_epochs, config=config, device=device), "test": "held-out region"})
ood_df = pd.DataFrame(ood_rows)
display(english_metric_table(ood_df)[["Model", "test", "Trajectory RMSE", "AUROC"]].round(4))
sl_df = ood_df[ood_df["test"] == "short→long"]; rg_df = ood_df[ood_df["test"] == "held-out region"]
grouped_bar(sl_df["model"].tolist(),
            {"short → long horizon": sl_df["Traj_RMSE"].tolist(), "held-out region": rg_df["Traj_RMSE"].tolist()},
            title="Out-of-distribution trajectory RMSE (lower is better)", ylabel="Trajectory RMSE (–)",
            save=SAVE_FIGS, fig_id="3_2_ood_rmse").show()

[TCN] эпоха 01 | обучение=0.2405 | валидация=-0.8089


[TCN] эпоха 02 | обучение=-0.6586 | валидация=-1.6763


[TCN] эпоха 01 | обучение=0.2291 | валидация=-0.4626


[TCN] эпоха 02 | обучение=-1.1053 | валидация=-1.7612


[DPI-Flow] эпоха 01 | обучение=-1.9328 | валидация=-2.3172


...

[EVT-NeuralSSM] эпоха 01 | обучение=-0.3648 | валидация=-0.9333


[EVT-NeuralSSM] эпоха 02 | обучение=-0.9888 | валидация=-0.9472


[EVT-NeuralSSM] эпоха 01 | обучение=-0.5507 | валидация=-0.9306


[EVT-NeuralSSM] эпоха 02 | обучение=-0.9978 | валидация=-1.0622


,Model,test,Trajectory RMSE,AUROC
0,TCN,short→long,0.1305,0.8846
1,TCN,held-out region,0.1620,0.8784
2,DPI-Flow,short→long,0.0477,0.9612
3,DPI-Flow,held-out region,0.0658,0.9499
4,EVT-NeuralSSM,short→long,0.1919,0.8049
5,EVT-NeuralSSM,held-out region,0.2154,0.6813


## Итог

Абляции подтверждают вклад компонентов; структурированные модели устойчивее в OOD.
Дальше — **3.3 разбор кейсов**.